# Baseline weekly rent

Same check as the [example rent report](https://www.microburbs.com.au/p/hackathon-example-rent-report).

Earlier adverts, 1 May 2025 to 30 April 2026, set the estimate. Later adverts, 1 May 2026 to 31 July 2026, score it. Those later adverts are not used to build or tune the groups.

Each group is suburb, bedrooms, and parking. The estimate is the median weekly asking rent. A group needs at least 10 earlier adverts. Smaller groups fall back to suburb and bedrooms, then bedrooms alone. Adverts missing bedrooms or parking stay out.

The published result on this check is a **$100** typical miss, **R² 0.645**, and **56.8%** within $100 (2,246 of 3,954).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path(".").resolve()
TRAIN_START = pd.Timestamp("2025-05-01")
TRAIN_END = pd.Timestamp("2026-04-30")
TEST_START = pd.Timestamp("2026-05-01")
TEST_END = pd.Timestamp("2026-07-31")
MIN_ADS = 10

leases = pd.read_csv(DATA / "rents-past-leases-city-of-sydney.csv")
leases["listing_date"] = pd.to_datetime(leases["listing_date"])
leases["bedrooms"] = pd.to_numeric(leases["bedrooms"], errors="coerce")
leases["car_space"] = pd.to_numeric(leases["car_space"], errors="coerce")
leases["weekly_rent"] = pd.to_numeric(leases["weekly_rent"], errors="coerce")
leases.shape

(112457, 15)

In [2]:
in_train = leases["listing_date"].between(TRAIN_START, TRAIN_END)
in_test = leases["listing_date"].between(TEST_START, TEST_END)
known = leases["bedrooms"].notna() & leases["car_space"].isin([0, 1])

train = leases.loc[in_train & known].copy()
test = leases.loc[in_test & known].copy()

print(f"Earlier adverts used: {len(train):,}")
print(f"Earlier adverts left out, missing bedrooms or parking: {(in_train & ~known).sum():,}")
print(f"Later adverts scored: {len(test):,}")
print(f"Later adverts left out, missing bedrooms or parking: {(in_test & ~known).sum():,}")

Earlier adverts used: 13,699
Earlier adverts left out, missing bedrooms or parking: 698
Later adverts scored: 3,954
Later adverts left out, missing bedrooms or parking: 129


In [3]:
def group_median(frame, keys):
    grouped = frame.groupby(keys, observed=True)["weekly_rent"].agg(median="median", n="size")
    return grouped.loc[grouped["n"] >= MIN_ADS, "median"]

by_suburb_bed_park = group_median(train, ["suburb", "bedrooms", "car_space"])
by_suburb_bed = group_median(train, ["suburb", "bedrooms"])
by_bed = group_median(train, ["bedrooms"])

print(f"Groups with at least {MIN_ADS} earlier adverts")
print(f"  suburb, bedrooms, parking: {len(by_suburb_bed_park):,}")
print(f"  suburb, bedrooms: {len(by_suburb_bed):,}")
print(f"  bedrooms: {len(by_bed):,}")

Groups with at least 10 earlier adverts
  suburb, bedrooms, parking: 152
  suburb, bedrooms: 103
  bedrooms: 5


In [4]:
scored = test.copy()
scored["estimate"] = [
    by_suburb_bed_park.get((suburb, beds, park), np.nan)
    for suburb, beds, park in zip(scored["suburb"], scored["bedrooms"], scored["car_space"])
]
scored["source"] = np.where(scored["estimate"].notna(), "suburb, bedrooms, parking", "")

need = scored["estimate"].isna()
scored.loc[need, "estimate"] = [
    by_suburb_bed.get((suburb, beds), np.nan)
    for suburb, beds in zip(scored.loc[need, "suburb"], scored.loc[need, "bedrooms"])
]
scored.loc[need & scored["estimate"].notna(), "source"] = "suburb, bedrooms"

need = scored["estimate"].isna()
scored.loc[need, "estimate"] = scored.loc[need, "bedrooms"].map(by_bed)
scored.loc[need & scored["estimate"].notna(), "source"] = "bedrooms"

print(scored["source"].value_counts().to_string())
print(f"Later adverts with no estimate: {scored['estimate'].isna().sum():,}")

source
suburb, bedrooms, parking    3845
suburb, bedrooms               72
bedrooms                       37
Later adverts with no estimate: 0


In [5]:
scored["miss"] = (scored["estimate"] - scored["weekly_rent"]).abs()
y = scored["weekly_rent"]
p = scored["estimate"]
r2 = 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
within = scored["miss"] <= 100

ours = {
    "Earlier adverts": f"{len(train):,}",
    "Later adverts": f"{len(scored):,}",
    "Typical miss": f"${scored['miss'].median():,.0f}",
    "Within $100": f"{within.mean():.1%} ({within.sum():,} of {len(scored):,})",
    "R squared": f"{r2:.3f}",
    "Used suburb, bedrooms, parking": f"{(scored['source'] == 'suburb, bedrooms, parking').sum():,}",
    "Used suburb, bedrooms": f"{(scored['source'] == 'suburb, bedrooms').sum():,}",
    "Used bedrooms": f"{(scored['source'] == 'bedrooms').sum():,}",
    "Typical miss, no parking": f"${scored.loc[scored['car_space'] == 0, 'miss'].median():,.0f}",
    "Typical miss, with parking": f"${scored.loc[scored['car_space'] == 1, 'miss'].median():,.0f}",
}
published = {
    "Earlier adverts": "13,699",
    "Later adverts": "3,954",
    "Typical miss": "$100",
    "Within $100": "56.8% (2,246 of 3,954)",
    "R squared": "0.645",
    "Used suburb, bedrooms, parking": "3,845",
    "Used suburb, bedrooms": "72",
    "Used bedrooms": "37",
    "Typical miss, no parking": "$85",
    "Typical miss, with parking": "$100",
}
pd.DataFrame({"Published": published, "This run": ours})

,Published,This run
Earlier adverts,"13,699","13,699"
Later adverts,"3,954","3,954"
Typical miss,$100,$100
Within $100,"56.8% (2,246 of 3,954)","56.8% (2,246 of 3,954)"
R squared,0.645,0.645
"Used suburb, bedrooms, parking","3,845","3,845"
"Used suburb, bedrooms",72,72
Used bedrooms,37,37
"Typical miss, no parking",$85,$85
"Typical miss, with parking",$100,$100


## What parking changes

The estimate above already separates each suburb and bedroom count by car space. The next cells remove that split, then put it back, on the same 3,954 later adverts.

The simple estimate uses every earlier advert that has a bedroom count. The parking estimate uses the 13,699 that also have a parking answer. Neither estimate sees the later adverts.

In [6]:
bed_known = leases["bedrooms"].notna()
simple_train = leases.loc[in_train & bed_known].copy()
simple_suburb_bed = group_median(simple_train, ["suburb", "bedrooms"])
simple_bed = group_median(simple_train, ["bedrooms"])

simple = test.copy()
simple["estimate"] = [
    simple_suburb_bed.get((suburb, beds), np.nan)
    for suburb, beds in zip(simple["suburb"], simple["bedrooms"])
]
need = simple["estimate"].isna()
simple.loc[need, "estimate"] = simple.loc[need, "bedrooms"].map(simple_bed)
simple["miss"] = (simple["estimate"] - simple["weekly_rent"]).abs()

gaps = []
for _, group in train.groupby(["suburb", "bedrooms"]):
    with_parking = group.loc[group["car_space"] == 1, "weekly_rent"]
    without_parking = group.loc[group["car_space"] == 0, "weekly_rent"]
    if len(with_parking) >= MIN_ADS and len(without_parking) >= MIN_ADS:
        gaps.append(with_parking.median() - without_parking.median())

print(f"Earlier adverts in the simple estimate: {len(simple_train):,}")
print(f"Comparable groups with both parking choices: {len(gaps):,}")
print(f"Typical parking gap: ${np.median(gaps):,.0f} a week")
print(f"Later adverts with no simple estimate: {simple['estimate'].isna().sum():,}")

Earlier adverts in the simple estimate: 14,331
Comparable groups with both parking choices: 53
Typical parking gap: $90 a week
Later adverts with no simple estimate: 0


In [7]:
def check_row(frame):
    y = frame["weekly_rent"]
    p = frame["estimate"]
    miss = frame["miss"]
    within = miss <= 100
    r2 = 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return {
        "Later adverts": f"{len(frame):,}",
        "Typical miss": f"${miss.median():,.2f}",
        "Within $100": f"{within.mean():.1%} ({within.sum():,} of {len(frame):,})",
        "R squared": f"{r2:.3f}",
        "Typical miss, no parking": f"${miss[frame['car_space'] == 0].median():,.2f}",
        "Typical miss, with parking": f"${miss[frame['car_space'] == 1].median():,.2f}",
    }

pd.DataFrame({
    "Suburb and bedrooms": check_row(simple),
    "Add parking": check_row(scored),
})

,Suburb and bedrooms,Add parking
Later adverts,"3,954","3,954"
Typical miss,$100.00,$100.00
Within $100,"55.2% (2,183 of 3,954)","56.8% (2,246 of 3,954)"
R squared,0.616,0.645
"Typical miss, no parking",$85.00,$85.00
"Typical miss, with parking",$100.00,$100.00


## Shortening the miss

Everything below keeps the same date split. Groups are built from adverts dated by 30 April 2026 and scored on the same 3,954 later adverts, which are never used to build or tune anything.

The suburb estimate gives every flat in a suburb the same rent. Inside one building and bedroom count the middle spread of asking rents is about $98 a week, against about $179 inside a suburb group, so the building is the sharper comparison. The catch is coverage: in the recent year only a quarter of the later adverts have 10 earlier adverts in their own building.

Two changes fix that. Drop the group size floor for buildings to one advert, and reach further back. Older asks cannot be pooled as they stand, so each one is brought forward to 30 April 2026 by how the city-wide quarterly median moved, then weighted down as it ages.

The settings below, the three year reach, the three year half life and the one advert floor, were chosen on three earlier validation folds ending in January 2026, October 2025 and July 2025. The May to July adverts were not looked at while choosing them.

In [8]:
import rent_model as rm

AS_AT = pd.Timestamp("2026-04-30")
usable = rm.load_leases(".")
earlier = usable.loc[usable["listing_date"] <= AS_AT]

building_only = rm.RentModel(as_at=AS_AT, max_age_years=1, half_life_years=None, neighbours=0, **rm.BUILDING_ONLY).fit(earlier)
no_neighbours = rm.RentModel(as_at=AS_AT, neighbours=0, **rm.BUILDING_ONLY).fit(earlier)
with_neighbours = rm.RentModel(as_at=AS_AT, shrink_adverts=0, distance_half_m=None, **rm.BUILDING_ONLY).fit(earlier)
unshrunk = rm.RentModel(as_at=AS_AT, shrink_adverts=0, distance_half_m=None).fit(earlier)
full = rm.RentModel(as_at=AS_AT).fit(earlier)

print(f"Adverts in the recent year: {building_only.adverts_used:,}")
print(f"Adverts once older asks are brought forward: {full.adverts_used:,}")
print(f"Typical parking gap used where a building advertised one side only: ${full.gap_typical:,.0f} a week")

Adverts in the recent year: 13,749
Adverts once older asks are brought forward: 39,683
Typical parking gap used where a building advertised one side only: $76 a week


In [9]:
steps = {
    "Suburb and bedrooms": check_row(simple),
    "Add parking": check_row(scored),
    "Same building, recent year": rm.check(test, building_only.predict(test)),
    "Older asks brought forward": rm.check(test, no_neighbours.predict(test)),
    "Plus nearest buildings": rm.check(test, with_neighbours.predict(test)),
    "Plus own past asks and bathrooms": rm.check(test, unshrunk.predict(test)),
    "Plus thin buildings pulled toward neighbours": rm.check(test, full.predict(test)),
}
pd.DataFrame(steps)

,Suburb and bedrooms,Add parking,"Same building, recent year",Older asks brought forward,Plus nearest buildings,Plus own past asks and bathrooms,Plus thin buildings pulled toward neighbours
Later adverts,"3,954","3,954","3,954","3,954","3,954","3,954","3,954"
Typical miss,$100.00,$100.00,$60.00,$55.71,$52.29,$45.00,$45.44
Within $100,"55.2% (2,183 of 3,954)","56.8% (2,246 of 3,954)","68.5% (2,709 of 3,954)","70.9% (2,804 of 3,954)","72.2% (2,855 of 3,954)","77.1% (3,050 of 3,954)","77.2% (3,053 of 3,954)"
R squared,0.616,0.645,0.782,0.821,0.834,0.872,0.871
"Typical miss, no parking",$85.00,$85.00,$50.00,$48.77,$49.41,$38.07,$37.97
"Typical miss, with parking",$100.00,$100.00,$70.00,$63.95,$60.00,$50.00,$50.00


### Where each estimate came from, and what is left over

The fallbacks matter as much as the headline. A later advert whose building has never advertised still has to get a number.

In [10]:
by_source = test.assign(
    estimate=full.predict(test),
    came_from=full.predict_both(test)["source"],
)
by_source["miss"] = (by_source["estimate"] - by_source["weekly_rent"]).abs()

summary = by_source.groupby("came_from").agg(
    later_adverts=("miss", "size"),
    typical_miss=("miss", "median"),
    within_100=("miss", lambda m: (m <= 100).mean()),
)
summary["typical_miss"] = summary["typical_miss"].map("${:,.2f}".format)
summary["within_100"] = summary["within_100"].map("{:.1%}".format)
summary.sort_values("later_adverts", ascending=False)

,later_adverts,typical_miss,within_100
came_from,,,
"building advertised both, bathrooms, own adverts",954,$33.34,88.1%
"building advertised both, bathrooms",802,$50.56,74.2%
"building, parking gap, bathrooms, own adverts",740,$30.70,88.4%
"building, parking gap, bathrooms",666,$52.83,72.2%
nearby buildings,234,$130.13,43.2%
"building, parking gap",212,$78.28,57.5%
"building, parking gap, own adverts",148,$43.31,80.4%
"nearby buildings, own adverts",93,$50.00,79.6%
building advertised both,51,$77.05,54.9%


### Floor level

`floor_level` is known for 34,396 of the 65,777 flats. Tested as a band adjustment on the three validation folds, it raised the typical miss on two of them, and inside building groups the leftover ratio barely moves with height. The building comparison already carries the floor, because a flat's own earlier advert usually sits in its group. It is not used.

In [11]:
flats = rm.load_flats(".")
with_floor = test.merge(flats[["gnaf_id", "floor_level"]], on="gnaf_id", how="left")
built = full.predict_both(with_floor)
leftover = with_floor.assign(
    ratio=with_floor["weekly_rent"] / full.predict(with_floor),
    came_from=built["source"],
)
leftover = leftover[leftover["came_from"].str.startswith("building") & leftover["floor_level"].notna()]

bands = pd.cut(leftover["floor_level"], [0, 3, 6, 11, 21, 100], right=False)
leftover.groupby(bands, observed=True)["ratio"].agg(later_adverts="size", typical_ratio="median").round(3)

,later_adverts,typical_ratio
floor_level,,
"[0, 3)",483,0.992
"[3, 6)",547,0.990
"[6, 11)",389,1.000
"[11, 21)",268,1.004
"[21, 100)",259,0.998


### Both rents, for every flat

The hand-in needs a rent with a car space and a rent without for all 65,777 flats, including the 16,094 that have never been advertised. Those come from their building's other flats.

When a building has only ever advertised one parking answer, the other rent is that one plus the parking gap from comparable flats in the same suburb and bedroom count. Where a building advertised both, its own two medians are used even if the flats with parking ask less, because that is what the adverts say.

This cell shows the rule applied as at 30 April 2026. The hand-in itself is written by `make_rents.py`, which refits on every advert dated by 31 July 2026.

In [12]:
both = full.predict_both(flats)
gap = both["rent_with_car_space"] - both["rent_without_car_space"]
never_advertised = flats["ever_advertised_for_lease"] == 0

print(f"Flats given both rents: {len(both):,}")
print(f"Flats that have never been advertised: {never_advertised.sum():,}")
print(f"Blank rents: {both[['rent_with_car_space', 'rent_without_car_space']].isna().sum().sum()}")
print(f"Typical parking gap across all flats: ${gap.median():,.2f} a week")
print(f"Flats where the adverts put parking lower: {(gap < 0).sum():,}")
print()
print(both["source"].value_counts().to_string())

Flats given both rents: 65,777
Flats that have never been advertised: 16,094
Blank rents: 0
Typical parking gap across all flats: $75.57 a week
Flats where the adverts put parking lower: 6,461

source
building advertised both, bathrooms                 15124
building, parking gap, bathrooms                    13427
building advertised both, bathrooms, own adverts    13227
building, parking gap, bathrooms, own adverts       11358
building, parking gap                                4634
nearby buildings                                     3412
building, parking gap, own adverts                   2741
building advertised both                             1014
building advertised both, own adverts                 694
nearby buildings, parking gap                         133
nearest bedroom count                                  11
bedrooms, parking gap                                   2
